# diagonal-via-strides — worked example 2: Extract the anti-diagonal via as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `diagonal-via-strides`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The anti-diagonal of a row-major `(N, N)` tensor runs from the top-right `m[0, N-1]` to the bottom-left `m[N-1, 0]`. Each step moves one row **down** (`+N`) and one column **left** (`-1`), so the diagonal-step stride is `N - 1`. The walk starts at `m[0, N-1]`, whose linear offset is `N - 1`, and has length `N`.

## Worked solution

**Step 1 — describe the anti-diagonal.** Its entries are `m[0, N-1], m[1, N-2], ..., m[N-1, 0]`. Unlike the main diagonal, the column index *decreases* as the row index increases.

**Step 2 — derive the stride.** One step down a row adds `N` to the linear index; one step left a column subtracts `1`. The net per-step change is `N - 1`, so `stride = (N - 1,)`. This is the key difference from every main/off-diagonal case, where the column moves right and the stride is `N + 1`.

**Step 3 — find the start offset.** The first element is `m[0, N-1]`, which sits at linear index `0 * N + (N - 1) = N - 1`. So `storage_offset = N - 1`.

**Step 4 — length.** The anti-diagonal of a square matrix has exactly `N` elements, so `size = (N,)`. We can confirm the walk stays in bounds: after `N - 1` steps the offset is `(N - 1) + (N - 1)*(N - 1) = (N-1)*N`, which is `m[N-1, 0]` — exactly the bottom-left corner, in range.

**Step 5 — verify.** PyTorch has no direct anti-diagonal helper, so we check against `torch.diagonal` applied to a left-right flipped copy: `t.diagonal(t.fliplr(m))` reads `m[0, N-1], m[1, N-2], ...`, the same sequence. Because `as_strided` returns a view, an in-place write through it mutates `m`.

In [ ]:
def anti_diagonal(m: Tensor) -> Tensor:
    N = m.shape[0]
    return m.as_strided(size=(N,), stride=(N - 1,), storage_offset=N - 1)


t.manual_seed(0)
N = 4
m = t.arange(N * N).reshape(N, N)
ad = anti_diagonal(m)
ref = t.diagonal(t.fliplr(m))
print("anti-diagonal:", ad.tolist())
print("matches fliplr+diagonal:", bool(t.equal(ad, ref)))
ad[0] = 999
print("view aliases m (m[0, N-1] mutated):", m[0, N - 1].item() == 999)